In [ ]:

import sys
print("Python version:", sys.version)


Python version: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


In [ ]:

from google.colab import files

uploaded = files.upload()


Saving 2026G3-main.zip to 2026G3-main.zip


In [ ]:

import zipfile
import os

zip_name = next(
    name for name in uploaded
    if name.endswith(".zip")
)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/tbda_project")

print("ZIP extracted successfully!")
print(os.listdir("/content/tbda_project"))


ZIP extracted successfully!
['2026G3-main']


In [ ]:

import os

project_dir = "/content/tbda_project/2026G3-main"

for root, dirs, files in os.walk(project_dir):
    for filename in files:
        if (
            "T37" in filename
            or "GBM_forallzones" in filename
            or "hybrid_blend" in filename
            or "T38" in filename
        ):
            print(os.path.relpath(
                os.path.join(root, filename),
                project_dir
            ))


cleaning_and_eda/T37_check_gbm_leakage.py
models/GBM_forallzones.ipynb
models/hybrid_blend_learned_weights.ipynb
models/T37_multihorizon_forecast.py
docs_and_slides/T37_REPORT.md
docs_and_slides/T37_LOGBOOK_ENTRY_and_BACKLOG_UPDATE.md
predictions_and_eval/T37_multihorizon_results.xlsx
predictions_and_eval/TeamB_T38_final_evaluation.xlsx
predictions_and_eval/T37_check_gbm_leakage_results.csv
predictions_and_eval/T37_experiment_log_rows.csv
predictions_and_eval/TeamB_T38_figures.py
predictions_and_eval/T37_multihorizon_predictions.csv.gz
predictions_and_eval/TeamB_T38_final_evaluation.py


In [ ]:

from pathlib import Path

script = Path(project_dir) / "cleaning_and_eda/T37_check_gbm_leakage.py"

print(script.read_text(encoding="utf-8"))


"""
T37 side check - is the day-ahead GBM of GBM_forallzones.ipynb leak-free?
(TBDA DataPack, CeDInt floor 1)  Author: Riccardo Tardanico

Why: T37 needs a day-ahead (1-day) reference to measure how accuracy decays with the
horizon. Before using the GBM of GBM_forallzones.ipynb as that reference, its features
are checked for information that would NOT be known when the forecast is issued.

Findings checked here (NW HVAC, same data, same model settings as the notebook):
  1. roll_mean_24 = rolling(24).mean() at hour t INCLUDES y(t) itself (target leakage).
  2. Lags are shifted by ROW, not by time; the series is complete on the hourly grid
     inside each period, so this only matters across the 902-day gap (lags of the first
     week of 2025 come from February 2023). Minor, not tested separately.
  3. temp_c, temp_lag1, temp_lag2, occupancy, lum_lux are indoor readings of hour t/t-1/t-2,
     which are not known 24 h ahead.
The script refits the notebook model three ways and prints CV

In [ ]:

import subprocess
import sys
from pathlib import Path

dataset = list(Path(project_dir).rglob(
    "T32_cedint_f1_model_ready.csv.gz"
))

print("Dataset trovati:", dataset)

assert len(dataset) == 1, "Controllare il percorso del dataset"

script = Path(project_dir) / "cleaning_and_eda/T37_check_gbm_leakage.py"

result = subprocess.run(
    [sys.executable, str(script), str(dataset[0])],
    cwd=project_dir,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)


Dataset trovati: [PosixPath('/content/tbda_project/2026G3-main/data/T32_cedint_f1_model_ready.csv.gz')]
                 variant  test    n  CV_RMSE_pct  NMBE_pct
        A notebook as is  2023  802         29.6       4.5
        A notebook as is  2025 1160         85.0       1.0
B roll_mean shifted 24 h  2023  802         49.6      16.0
B roll_mean shifted 24 h  2025 1160         97.6      -0.6
       C fully day-ahead  2023  802         49.5      14.3
       C fully day-ahead  2025 1160         98.4     -13.5


Return code: 0


In [ ]:

import pandas as pd

df = pd.read_csv(dataset[0])

print("Numero di righe:", len(df))
print("\nColonne disponibili:")
print(df.columns.tolist())

print("\nSerie energetiche:")
series = (
    df[["zone", "load_type"]]
    .drop_duplicates()
    .sort_values(["zone", "load_type"])
)

print(series.to_string(index=False))

print("\nPeriodi disponibili:")
print(df["split"].value_counts(dropna=False))


Numero di righe: 470680

Colonne disponibili:
['timestamp_utc', 'timestamp_local', 'zone', 'zone_code', 'load_type', 'n_devices', 'era', 'split', 'model_scope', 'recommendation', 'series_caution', 'level_shift_2025', 'include_train', 'include_train_no2020', 'include_eval', 'exclusion_reason', 'energy_wh_model', 'energy_source', 'hour_local', 'dayofweek_local', 'month_local', 'is_weekend', 'is_holiday', 'is_august_low_use', 'flag_covid_2020', 'temp_c', 'hum_pct', 'lum_lux', 'occupancy', 'lag_24h', 'lag_168h']

Serie energetiche:
zone       load_type
  NE            HVAC
  NE Plugs & Sockets
  NW Emerg. Lighting
  NW            HVAC
  NW      Hand Dryer
  NW        Lighting
  NW Plugs & Sockets
  NW            RACK
  SE Emerg. Lighting
  SE            HVAC
  SE        Lighting
  SE Plugs & Sockets
  SE       Projector
  SW Emerg. Lighting
  SW      Hand Dryer
  SW        Lighting
  SW Plugs & Sockets
  SW            RACK

Periodi disponibili:
split
train         323298
validation     750

In [ ]:

from pathlib import Path
import pandas as pd

# Troviamo il file della selezione finale T38
selection_files = list(
    Path(project_dir).rglob("best_model_selection_final.csv")
)

print("File trovati:", selection_files)

if len(selection_files) == 1:
    selection = pd.read_csv(selection_files[0])

    print("\nColonne:")
    print(selection.columns.tolist())

    print("\nSelezione originale:")
    display(selection.head(20))
else:
    print("Controllare il percorso del file.")


File trovati: [PosixPath('/content/tbda_project/2026G3-main/predictions_and_eval/best_model_selection_final.csv')]

Colonne:
['zone', 'load_type', 'best_model', 'candidates', 'selection_hours', 'sel_cv_best', 'sel_cv_naive_day', 'sel_cv_catboost', 'sel_cv_catboost_no2020', 'sel_cv_chronos_bolt_base', 'sel_cv_gbm', 'sel_cv_hybrid_chronos_gbm_learned_weights', 'sel_cv_lightgbm', 'sel_cv_random_forest', 'sel_cv_random_forest_no2020']

Selezione originale:


,zone,load_type,best_model,candidates,selection_hours,sel_cv_best,sel_cv_naive_day,sel_cv_catboost,sel_cv_catboost_no2020,sel_cv_chronos_bolt_base,sel_cv_gbm,sel_cv_hybrid_chronos_gbm_learned_weights,sel_cv_lightgbm,sel_cv_random_forest,sel_cv_random_forest_no2020
0,NE,HVAC,gbm,8,1813,325.35,461.09,325.86,325.84,325.76,325.35,325.42,325.86,325.82,325.73
1,NE,Plugs & Sockets,gbm,8,1813,23.24,50.33,57.84,64.05,42.78,23.24,34.16,57.70,54.52,56.07
2,NW,Emerg. Lighting,chronos_bolt_base,5,1630,2.70,3.46,8.34,9.43,2.70,NaN,NaN,NaN,5.20,11.21
3,NW,HVAC,chronos_bolt_base,8,1606,19.12,23.55,34.20,31.56,19.12,31.25,34.93,28.35,28.40,27.73
4,NW,Lighting,hybrid_chronos_gbm_learned_weights,8,1813,64.35,93.14,90.11,87.69,65.45,84.82,64.35,90.39,111.24,90.27
5,NW,Plugs & Sockets,gbm,8,1813,37.88,63.99,47.96,48.02,48.76,37.88,52.83,49.17,48.96,48.49
6,NW,RACK,chronos_bolt_base,5,1837,0.74,1.03,8.15,15.46,0.74,NaN,NaN,NaN,1.69,1.76
7,SE,Emerg. Lighting,chronos_bolt_base,5,1612,12.74,19.93,27.78,27.68,12.74,NaN,NaN,NaN,27.52,27.54
8,SE,HVAC,chronos_bolt_base,8,1136,21.86,41.27,44.70,45.25,21.86,36.38,26.12,42.74,42.65,43.46
9,SE,Lighting,hybrid_chronos_gbm_learned_weights,8,1071,193.67,373.34,292.77,257.19,263.63,248.49,193.67,314.54,301.30,288.93


In [ ]:

from pathlib import Path

pred_dir = Path(project_dir) / "predictions_and_eval"

for f in sorted(pred_dir.rglob("*")):
    if f.is_file() and (
        "predict" in f.name.lower()
        or "forecast" in f.name.lower()
        or "selection" in f.name.lower()
    ):
        print(f.relative_to(project_dir))


predictions_and_eval/All_Zones_GBM_predictions.csv
predictions_and_eval/All_Zones_HYBRID_BLEND_predictions.csv
predictions_and_eval/NW_HVAC_GBM_predictions.csv
predictions_and_eval/T31_NE_predictions.csv
predictions_and_eval/T36_combine_predictions.py
predictions_and_eval/T37_multihorizon_predictions.csv.gz
predictions_and_eval/best_model_selection_final.csv


In [ ]:

import pandas as pd
from pathlib import Path

pred_dir = Path(project_dir) / "predictions_and_eval"

files_to_check = [
    "All_Zones_GBM_predictions.csv",
    "All_Zones_HYBRID_BLEND_predictions.csv"
]

for filename in files_to_check:
    path = pred_dir / filename
    data = pd.read_csv(path)

    print("\nFILE:", filename)
    print("Righe:", len(data))
    print("Colonne:", data.columns.tolist())
    print("\nPrime 3 righe:")
    display(data.head(3))

    if "split" in data.columns:
        print("\nPeriodi:")
        print(data["split"].value_counts())



FILE: All_Zones_GBM_predictions.csv
Righe: 48221
Colonne: ['timestamp_utc', 'zone', 'load_type', 'model', 'split', 'y_true', 'y_pred']

Prime 3 righe:


,timestamp_utc,zone,load_type,model,split,y_true,y_pred
0,2022-07-01 02:00:00,NW,HVAC,gbm,test,157.2905,152.076103
1,2022-07-01 03:00:00,NW,HVAC,gbm,test,156.4920,154.495634
2,2022-07-01 04:00:00,NW,HVAC,gbm,test,156.9405,157.355841



Periodi:
split
test    48221
Name: count, dtype: int64

FILE: All_Zones_HYBRID_BLEND_predictions.csv
Righe: 48040
Colonne: ['timestamp_utc', 'zone', 'load_type', 'model', 'split', 'y_true', 'y_pred']

Prime 3 righe:


,timestamp_utc,zone,load_type,model,split,y_true,y_pred
0,2022-07-01 00:00:00,NE,HVAC,hybrid_chronos_gbm_learned_weights,validation,2.1355,2.503666
1,2022-07-01 00:00:00,NE,Plugs & Sockets,hybrid_chronos_gbm_learned_weights,validation,188.5730,188.420918
2,2022-07-01 00:00:00,NW,HVAC,hybrid_chronos_gbm_learned_weights,validation,157.2905,156.457330



Periodi:
split
validation    30160
test_2023      9728
test_2025      8152
Name: count, dtype: int64


In [ ]:

from pathlib import Path

print("File GBM presenti nel progetto:\n")

for path in Path(project_dir).rglob("*GBM*"):
    if path.is_file():
        print(path.relative_to(project_dir))

print("\nFile con 'utc' nel nome:\n")

for path in Path(project_dir).rglob("*utc*"):
    if path.is_file() and "prediction" in path.name.lower():
        print(path.relative_to(project_dir))


File GBM presenti nel progetto:

figures/NW_HVAC_GBM_Grafiek.png
models/GBM_forallzones.ipynb
predictions_and_eval/All_Zones_GBM_ASHRAE_metrics.xlsx
predictions_and_eval/NW_HVAC_GBM_predictions.csv
predictions_and_eval/All_Zones_GBM_predictions.csv
predictions_and_eval/NW_HVAC_GBM_ASHRAE_metrics.xlsx

File con 'utc' nel nome:



In [ ]:

import json
from pathlib import Path

notebook_path = Path(project_dir) / "models/hybrid_blend_learned_weights.ipynb"

with open(notebook_path, encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "tz_localize",
    "tz_convert",
    "Europe/Madrid",
    "All_Zones_GBM",
    "timestamp_utc"
]

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    if any(word in source for word in keywords):
        print(f"\n{'='*15} CELLA {i} {'='*15}")
        print(source[:5000])



=============== CELLA 0 ===============
# Hybrid blend with learned weights (Chronos + GBM)

**Answer to the customer question from the mid-term review:**
> *"Chronos reads the history of the series, the GBM reads the drivers. Why not combine the two? For example with a hybrid model?"*

`blend_model_chronos&gbm.ipynb` (Joos) combines the two parents with a **fixed** 50/50 average.
**This notebook** goes one step further: a **trained** blend. A linear stacker learns, per series, how much to trust each parent:

$$\hat{y} = w_{chronos} \cdot y^{chronos} + w_{gbm} \cdot y^{gbm} + b$$

The weights are not chosen by hand: they are estimated by ordinary least squares on the validation period, **per series**. On NW HVAC the model learns to follow the GBM (w_gbm ~ 0.8); on the lighting series it learns a balanced mix. That is the direct answer to the customer: *learn when to trust the univariate pattern and when to rely on the drivers.*

**Honest protocol** (same rules as `team_b_eval.py` and 